In [1]:
from pathlib import Path

from IPython.display import display

from searchlight.lightcurves import load
from searchlight.review_exports import (
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    enrich_results_with_metadata,
    load_lensing_results,
)
from searchlight.review_widgets import make_zooniverse_review_item
from searchlight.zooniverse_results import (
    DEFAULT_SUBJECT_METADATA_CACHE_PATH,
    DEFAULT_TASK_KEY,
    add_candidate_ids,
    annotate_most_likely_answers,
    load_zooniverse_results,
    select_zooniverse_answer_results,
)


In [2]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH

# CSV exported from Zooniverse/aggregation, containing integer subject_id and data.most_likely.
ZOONIVERSE_RESULTS_CSV = Path('data/zooniverse/check_peaks_results.csv')
SUBJECT_METADATA_CACHE_PATH = DEFAULT_SUBJECT_METADATA_CACHE_PATH

# Set this explicitly if the CSV does not contain a workflow_id column.
WORKFLOW_ID = None
TASK_KEY = DEFAULT_TASK_KEY
MOST_LIKELY_ANSWER = 'Three or more peaks'
TOP_N = 100
MIN_CLASSIFICATIONS = 3
MIN_AGREEMENT = 0.7


In [3]:
# Zooniverse CSV loading, workflow answer lookup, subject metadata mapping,
# and review-card rendering live in searchlight modules so notebooks and
# exporters share the same implementation.


In [4]:
# Review widget implementation: searchlight.review_widgets.make_zooniverse_review_item


In [5]:
results = load_lensing_results(
    results_path=RESULTS_PATH,
    cache_path=RESULT_CACHE_PATH,
    cache_meta_path=RESULT_CACHE_META_PATH,
)
results = add_candidate_ids(results)
print(f'Loaded {len(results)} lensing model rows from {RESULTS_PATH}')


Loading cached results from data/results/simulens1.0_tess_spoc/top_resnet_results_cache.pkl
Loaded 2160368 lensing model rows from data/results/simulens1.0_tess_spoc


In [6]:
zooniverse = load_zooniverse_results(ZOONIVERSE_RESULTS_CSV)
zooniverse, workflow_id, answer_labels = annotate_most_likely_answers(
    zooniverse,
    workflow_id=WORKFLOW_ID,
    task_key=TASK_KEY,
)
zooniverse['most_likely_answer'].value_counts(dropna=False)


Workflow 31708 T0 answers: {0: 'None, it just contains noise or stellar variability', 1: 'One peak', 2: 'Two peaks', 3: 'Three or more peaks'}


most_likely_answer
One peak                                               34031
None, it just contains noise or stellar variability    20400
NaN                                                     8085
Two peaks                                               4049
Three or more peaks                                     2215
Name: count, dtype: int64

In [7]:
selected = select_zooniverse_answer_results(
    results,
    zooniverse,
    MOST_LIKELY_ANSWER,
    MIN_CLASSIFICATIONS, 
    MIN_AGREEMENT,
    subject_metadata_cache_path=SUBJECT_METADATA_CACHE_PATH,
)
review_results = selected.head(TOP_N).copy()
review_results[['subject_id', 'most_likely_answer', 'data.num_votes', 'data.agreement', 'candidate_id', 'path', 'lensing']].head()


Fetching Zooniverse subjects: 0subject [00:00, ?subject/s]

Selected 35 rows where most_likely_answer == 'Three or more peaks'
Mapped 35 selected subjects to paths


,subject_id,most_likely_answer,data.num_votes,data.agreement,candidate_id,path,lensing
0,119505647,Three or more peaks,3,1.00,9f741e5f-eaad-5f7c-ad8a-ab4321d4f1ef,https://mast.stsci.edu/api/v0.1/Download/file/...,5.102989
1,119512525,Three or more peaks,3,0.75,bac62b59-152d-5943-9beb-c91c617177a4,https://mast.stsci.edu/api/v0.1/Download/file/...,4.697097
2,119526852,Three or more peaks,3,1.00,1557a46c-24fc-5da0-8ed6-7ceda653d2ae,https://mast.stsci.edu/api/v0.1/Download/file/...,4.331447
3,119529212,Three or more peaks,3,1.00,68e067cc-702c-5bfc-bc5d-f5a70be2ce6b,https://mast.stsci.edu/api/v0.1/Download/file/...,4.280339
4,119530411,Three or more peaks,3,1.00,e5aefd8a-62c3-52ba-b453-fdc67881cb98,https://mast.stsci.edu/api/v0.1/Download/file/...,4.250017


In [8]:
review_export = enrich_results_with_metadata(
    review_results,
    cache_path=RESULT_METADATA_CACHE_PATH,
)
review_export[[
    'subject_id',
    'most_likely_answer',
    'data.num_votes',
    'data.agreement',
    'path',
    'lensing',
    'tic',
    'sector',
    'simbad_url',
]].head()


,subject_id,most_likely_answer,data.num_votes,data.agreement,path,lensing,tic,sector,simbad_url
0,119505647,Three or more peaks,3,1.00,https://mast.stsci.edu/api/v0.1/Download/file/...,5.102989,395182982,38,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
1,119512525,Three or more peaks,3,0.75,https://mast.stsci.edu/api/v0.1/Download/file/...,4.697097,84369799,55,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
2,119526852,Three or more peaks,3,1.00,https://mast.stsci.edu/api/v0.1/Download/file/...,4.331447,432928699,36,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
3,119529212,Three or more peaks,3,1.00,https://mast.stsci.edu/api/v0.1/Download/file/...,4.280339,306107053,39,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
4,119530411,Three or more peaks,3,1.00,https://mast.stsci.edu/api/v0.1/Download/file/...,4.250017,138019165,58,https://simbad.cds.unistra.fr/simbad/sim-coo?C...


In [9]:
for row in review_results.to_dict('records'):
    lc = load(row['path'])
    display(make_zooniverse_review_item(row, lc))


0% (12/3705) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (12/3705) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (14/3879) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (14/3879) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (12/11704) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (12/11704) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


...

0% (26/10837) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (26/10837) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


1% (7/1093) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:1% (7/1093) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (16/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (16/3477) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3781) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
